# MRR

Monthly Recurring Revenue computed by the Tidemill engine.

## How MRR is normalised

MRR converts every billing interval to a **monthly** rate:

| Interval | Formula                              |
|----------|--------------------------------------|
| month    | `amount / interval_count`            |
| year     | `amount / (12 * interval_count)`     |
| week     | `amount * 52 / (12 * interval_count)`|
| day      | `amount * 365 / (12 * interval_count)`|

Only active subscriptions contribute to current MRR.
All arithmetic uses integer cents to avoid floating-point drift.

In [1]:
from tidemill import reports
from tidemill.reports.client import TidemillClient

reports.setup()

START, END = "2025-06-01", "2026-03-31"
tm = TidemillClient()

## 1. MRR Snapshot

Current Monthly Recurring Revenue and annualised ARR.

In [2]:
reports.mrr.style_snapshot(reports.mrr.snapshot(tm))

MRR,ARR
"$1,068.33","$12,819.96"


## 2. MRR Breakdown (movements)

Tidemill classifies every MRR change into a **movement type**:

| Movement      | Meaning                                              |
|---------------|------------------------------------------------------|
| **new**       | First subscription for a customer                    |
| **expansion** | Upgrade or additional subscription                   |
| **contraction** | Downgrade (lower plan/fewer seats)                 |
| **churn**     | Cancellation — customer lost entirely                |
| **reactivation** | Previously churned customer returns               |

The breakdown endpoint sums these over the full period.

In [3]:
reports.mrr.plot_breakdown(reports.mrr.breakdown(tm, START, END))

## 3. Quick Ratio

Growth efficiency ratio:

$$\text{Quick Ratio} = \frac{\text{new} + \text{expansion} + \text{reactivation}}{|\text{churn}| + |\text{contraction}|}$$

Interprets how much MRR is added for each dollar lost.

| Ratio | Interpretation                         |
|-------|----------------------------------------|
| > 4   | Excellent — strong, efficient growth   |
| 2 – 4 | Healthy growth                         |
| 1 – 2 | Slow growth                            |
| < 1   | Contracting — losses exceed gains      |

In [4]:
reports.mrr.style_quick_ratio(reports.mrr.quick_ratio(tm, START, END))

New,Expansion,Reactivation,Gains,Churn,Contraction,Losses,Quick Ratio
"$1,487.33",$118.00,$99.00,"$1,704.33",$558.00,$118.00,$676.00,2.52


## 4. Monthly MRR with movement split

Query MRR breakdown per month to show how new, expansion, contraction, and churn compose each month's MRR change.

In [5]:
wf = reports.mrr.waterfall(tm, START, END)
reports.mrr.style_waterfall(wf)

,starting_mrr,new,expansion,reactivation,contraction,churn,net_change,ending_mrr
month,,,,,,,,
2025-06,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00
2025-07,$0.00,"$1,007.33",$0.00,$0.00,$0.00,$-40.00,$967.33,$967.33
2025-08,$967.33,$80.00,$59.00,$0.00,$0.00,$-80.00,$59.00,"$1,026.33"
2025-09,"$1,026.33",$0.00,$0.00,$0.00,$-59.00,$-119.00,$-178.00,$848.33
2025-10,$848.33,$160.00,$0.00,$20.00,$0.00,$-40.00,$140.00,$988.33
2025-11,$988.33,$0.00,$59.00,$79.00,$-59.00,$-119.00,$-40.00,$948.33
2025-12,$948.33,$140.00,$0.00,$0.00,$0.00,$-20.00,$120.00,"$1,068.33"
2026-01,"$1,068.33",$100.00,$0.00,$0.00,$0.00,$-100.00,$0.00,"$1,068.33"
2026-02,"$1,068.33",$0.00,$0.00,$0.00,$0.00,$-40.00,$-40.00,"$1,028.33"


### 4a. Per-customer movement log

Drill into the individual movements that compose each month's waterfall.
Every row is a single customer's MRR change — **new**, **expansion**,
**reactivation**, **contraction**, or **churn** — with monthly subtotals
that should match the waterfall table above.

In [6]:
ml = reports.mrr.movement_log(tm, START, END)
reports.mrr.style_movement_log(ml)

month,date,customer,customer_id,movement,amount
2025-07,2025-07-01,Active Annual Enterprise #7,cus_ULeP4MdsGIqA3N,new,$207.50
2025-07,2025-07-01,Active Annual Pro #6,cus_ULeP2nHUy4PfEW,new,$65.83
2025-07,2025-07-01,Active Monthly Pro #4,cus_ULePdAg7TfFaRx,new,$79.00
2025-07,2025-07-01,Active Monthly Pro #5,cus_ULePZEpGaYDqub,new,$79.00
2025-07,2025-07-01,Active Starter #1,cus_ULeOmo28L85BDt,new,$20.00
2025-07,2025-07-01,Active Starter #2,cus_ULePYzNMidnEnZ,new,$20.00
2025-07,2025-07-01,Active Starter Heavy #3,cus_ULePUoQI3jKcEM,new,$20.00
2025-07,2025-07-01,Churned Pro #11,cus_ULeP0p3hQP4tXH,new,$79.00
2025-07,2025-07-01,Churned Starter #8,cus_ULePpEt37T6vDX,new,$20.00
2025-07,2025-07-01,Churn→Reactivate Pro #16,cus_ULeQx6mikUSGsG,new,$79.00


In [7]:
reports.mrr.plot_waterfall(wf)

In [8]:
reports.mrr.plot_trend(reports.mrr.trend(tm, START, END))